# Error Analysis — Image Classification

Run after `predict` (or `make sample-preds`): needs `data/run/predictions.parquet` + `data/run/classes.json`. The goal is a **failure taxonomy with evidence**, then a measured fix.

1. Load predictions + report
2. Confusion matrix
3. Most-confused pairs + weakest classes
4. Most-confident mistakes (inspect images)
5. Calibration
6. Failure taxonomy + conclusions

In [ ]:
import json
import numpy as np
import pandas as pd
from src.metrics import confusion_matrix, classification_report
from src.error_analysis import most_confused_pairs, per_class_error_rate, top_misclassified, calibration_bins

df = pd.read_parquet('data/run/predictions.parquet')
classes = json.load(open('data/run/classes.json'))
y_true, y_pred, conf = df.y_true.to_numpy(), df.y_pred.to_numpy(), df.confidence.to_numpy()
cm = confusion_matrix(y_true, y_pred, len(classes))
classification_report(cm, classes)

In [ ]:
from src.error_analysis import plot_confusion
plot_confusion(cm, classes, '/tmp/cm.png')
from PIL import Image
Image.open('/tmp/cm.png')

In [ ]:
print('Most confused (true -> pred, count):')
for i, j, c in most_confused_pairs(cm, 8):
    print(f'  {classes[i]:>12} -> {classes[j]:<12} {c}')

err = per_class_error_rate(cm)
print('\nWeakest classes:')
for i in np.argsort(-err)[:5]:
    print(f'  {classes[i]:>12}  error_rate={err[i]:.3f}')

In [ ]:
# The model's most confident mistakes — inspect the images
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

idx = top_misclassified(y_true, y_pred, conf, k=8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), idx):
    r = df.iloc[int(i)]
    if Path(r.path).exists():
        ax.imshow(Image.open(r.path).convert('RGB'))
    ax.set_title(f"{classes[int(r.y_true)]} -> {classes[int(r.y_pred)]}  ({r.confidence:.2f})", fontsize=9)
    ax.axis('off')
plt.tight_layout()

In [ ]:
bins, ece = calibration_bins(conf, (y_true == y_pred).astype(float))
print('ECE:', ece)
b = pd.DataFrame(bins)
ax = b.plot(x='confidence', y='accuracy', marker='o', legend=False, title=f'Reliability (ECE={ece})')
ax.plot([0, 1], [0, 1], '--', color='grey'); ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')

## Failure taxonomy + conclusions

| Category | Example (true → pred) | Count | Fix tried | Result (Δ macro-F1 / ECE) |
|---|---|---|---|---|
| Visually similar classes | | | bigger backbone / higher res | |
| Rare class under-recalled | | | oversample / focal loss | |
| Confidently wrong (poor calibration) | | | label smoothing / temp scaling | |
| Bad/ambiguous labels | | | relabel / drop | |
| Background focus (Grad-CAM) | | | better crops / augmentation | |

**Conclusion (fill in):** which intervention helped most, the measured lift, and why.